# HGT 从零实现：异构交易图风险识别

## 面试问题

面试时我会说明 HGT 不把所有节点和边塞进同一个投影。Query 由目标节点类型投影，Key/Value 由源节点类型投影，关系类型还拥有自己的 attention 和 message 变换；这样相同数值在用户、设备、订单中的含义不会混淆。不同关系产生的边分数必须在同一目标节点的全部入边上归一化，才能比较用户证据和设备证据。类型专属输出与残差保留目标节点自身信息。下面用订单—用户—设备图比较订单单点规则，手写关系感知注意力、真实反向传播，并复现错误的全图 softmax。

## 真实案例

图包含六个订单、四个用户和四台设备。每个订单各有一条用户下单边和一条设备使用边；订单标签来自离线人工调查。O002、O004 的订单自身字段不明显，但关联到高拒付用户和高风险共享设备。该小图省略了时间、更多关系和标签延迟。

本实验是用于解释机制的确定性小样本，所有指标均标记为“教学实验”，不能外推为线上收益。

In [1]:
import math  # 导入平方根用于缩放关系注意力。
import torch  # 导入 PyTorch 以实现异构图消息传递。
from torch import nn  # 导入神经网络基础模块。
import torch.nn.functional as F  # 导入激活和分类损失。
torch.manual_seed(49)  # 固定随机种子以复现实验输出。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小图训练。
order_ids = ["O001", "O002", "O003", "O004", "O005", "O006"]  # 定义六个脱敏订单编号。
user_ids = ["U01", "U02", "U03", "U04"]  # 定义四个脱敏用户编号。
device_ids = ["D01", "D02", "D03", "D04"]  # 定义四台脱敏设备编号。
node_features = {"order": torch.tensor([[0.80, 0.50, 0.20], [0.30, 0.40, 0.10], [0.60, 0.10, 0.20], [0.40, 0.40, 0.40], [0.70, 0.50, 0.30], [0.50, 0.20, 0.20]], dtype=torch.float32), "user": torch.tensor([[0.10, 0.90, 0.00], [0.80, 0.10, 1.00], [0.20, 0.80, 0.00], [0.70, 0.10, 1.00]], dtype=torch.float32), "device": torch.tensor([[0.90, 0.90, 1.00], [0.10, 0.10, 0.00], [0.80, 0.80, 1.00], [0.20, 0.10, 0.00]], dtype=torch.float32)}  # 保存订单、用户和设备各自三维语义特征。
order_labels = torch.tensor([1, 1, 0, 1, 1, 0], dtype=torch.long)  # 保存六个订单的人工风险标签。
edge_sets = {"user__places__order": torch.tensor([[0, 0, 1, 2, 2, 3], [0, 1, 2, 3, 4, 5]], dtype=torch.long), "device__used__order": torch.tensor([[0, 0, 1, 2, 2, 3], [0, 1, 2, 3, 4, 5]], dtype=torch.long)}  # 定义用户下单和设备使用两类有向边。
print("订单   订单特征(金额,改址,优惠)  用户  用户特征(资历,拒付,实名)  设备  设备特征(共享,风险,模拟器)  标签")  # 打印异构图输入表头。
for order_index, order_id in enumerate(order_ids):  # 逐订单展示三种节点及其关系。
    user_index = edge_sets["user__places__order"][0, order_index].item()  # 读取当前订单关联用户编号。
    device_index = edge_sets["device__used__order"][0, order_index].item()  # 读取当前订单关联设备编号。
    print(f"{order_id}  {node_features['order'][order_index].tolist()}  {user_ids[user_index]}  {node_features['user'][user_index].tolist()}  {device_ids[device_index]}  {node_features['device'][device_index].tolist()}  {order_labels[order_index].item()}")  # 输出当前订单的异构邻域证据。
print(f"节点数：订单={len(order_ids)}，用户={len(user_ids)}，设备={len(device_ids)}；关系边总数={sum(edges.shape[1] for edges in edge_sets.values())}")  # 汇总异构图规模。

订单   订单特征(金额,改址,优惠)  用户  用户特征(资历,拒付,实名)  设备  设备特征(共享,风险,模拟器)  标签
O001  [0.800000011920929, 0.5, 0.20000000298023224]  U01  [0.10000000149011612, 0.8999999761581421, 0.0]  D01  [0.8999999761581421, 0.8999999761581421, 1.0]  1
O002  [0.30000001192092896, 0.4000000059604645, 0.10000000149011612]  U01  [0.10000000149011612, 0.8999999761581421, 0.0]  D01  [0.8999999761581421, 0.8999999761581421, 1.0]  1
O003  [0.6000000238418579, 0.10000000149011612, 0.20000000298023224]  U02  [0.800000011920929, 0.10000000149011612, 1.0]  D02  [0.10000000149011612, 0.10000000149011612, 0.0]  0
O004  [0.4000000059604645, 0.4000000059604645, 0.4000000059604645]  U03  [0.20000000298023224, 0.800000011920929, 0.0]  D03  [0.800000011920929, 0.800000011920929, 1.0]  1
O005  [0.699999988079071, 0.5, 0.30000001192092896]  U03  [0.20000000298023224, 0.800000011920929, 0.0]  D03  [0.800000011920929, 0.800000011920929, 1.0]  1
O006  [0.5, 0.20000000298023224, 0.20000000298023224]  U04  [0.699999988079071, 0.100000001

## 基线：只读取订单自身字段

基线把金额强度和改址强度相加后设阈值，不读取用户或设备节点。它与 HGT 在相同六个订单上计算准确率。

In [2]:
baseline_scores = node_features["order"][:, 0] + node_features["order"][:, 1]  # 合并订单金额和改址两个单点风险字段。
baseline_predictions = (baseline_scores > 1.0).long()  # 用固定阈值得到不看异构邻居的预测。
baseline_accuracy = (baseline_predictions == order_labels).float().mean().item()  # 计算订单单点规则准确率。
print("订单   单点分数  基线预测  真实标签")  # 打印逐订单基线结果表头。
for order_index, order_id in enumerate(order_ids):  # 遍历六个订单观察规则错误。
    print(f"{order_id}  {baseline_scores[order_index]:.2f}      {baseline_predictions[order_index].item()}         {order_labels[order_index].item()}")  # 输出当前订单的分数与标签。
print(f"订单单点规则准确率={baseline_accuracy:.1%}")  # 汇总同数据上的基线指标。

订单   单点分数  基线预测  真实标签
O001  1.30      1         1
O002  0.70      0         1
O003  0.70      0         0
O004  0.80      0         1
O005  1.20      1         1
O006  0.70      0         0
订单单点规则准确率=66.7%


## 手写核心：节点类型投影、关系矩阵和按目标归一化

`HGTOrderLayer` 分别注册订单 Query、用户/设备 Key/Value 和两类关系矩阵。所有边分数拼接后，再按目标订单做 softmax，保证一个订单的用户边与设备边权重之和为 1。

In [3]:
class HGTOrderLayer(nn.Module):  # 定义专门聚合到订单节点的手写 HGT 层。
    def __init__(self, input_dim=3, hidden_dim=8):  # 根据输入和隐藏维度创建类型与关系参数。
        super().__init__()  # 初始化父类以注册全部参数。
        self.hidden_dim = hidden_dim  # 保存隐藏维度用于缩放注意力。
        self.query_projection = nn.Linear(input_dim, hidden_dim, bias=False)  # 创建订单类型专属 Query 投影。
        self.key_projections = nn.ModuleDict({"user": nn.Linear(input_dim, hidden_dim, bias=False), "device": nn.Linear(input_dim, hidden_dim, bias=False)})  # 创建用户与设备类型专属 Key 投影。
        self.value_projections = nn.ModuleDict({"user": nn.Linear(input_dim, hidden_dim, bias=False), "device": nn.Linear(input_dim, hidden_dim, bias=False)})  # 创建用户与设备类型专属 Value 投影。
        self.relation_attention = nn.ParameterDict({"user__places__order": nn.Parameter(torch.eye(hidden_dim)), "device__used__order": nn.Parameter(torch.eye(hidden_dim))})  # 为两类关系创建独立 attention 变换。
        self.relation_message = nn.ParameterDict({"user__places__order": nn.Parameter(torch.eye(hidden_dim)), "device__used__order": nn.Parameter(torch.eye(hidden_dim))})  # 为两类关系创建独立 message 变换。
        self.output_projection = nn.Linear(hidden_dim, hidden_dim)  # 创建订单聚合后的类型专属输出投影。
        self.residual_projection = nn.Linear(input_dim, hidden_dim)  # 创建订单自身特征的残差旁路。
    def forward(self, features_by_type, relations):  # 汇总两类关系到订单并返回逐边中间量。
        order_queries = self.query_projection(features_by_type["order"])  # 计算所有目标订单的 Query。
        all_scores = []  # 收集不同关系的逐边注意力分数。
        all_values = []  # 收集不同关系变换后的逐边消息。
        all_targets = []  # 收集每条消息的目标订单编号。
        all_metadata = []  # 收集每条边的关系名和源节点编号供解释。
        for relation_name, edges in relations.items():  # 逐关系类型计算专属 Key、Value 和分数。
            source_type = relation_name.split("__")[0]  # 从关系名读取源节点类型。
            source_indices = edges[0]  # 读取当前关系每条边的源节点编号。
            target_indices = edges[1]  # 读取当前关系每条边的目标订单编号。
            projected_keys = self.key_projections[source_type](features_by_type[source_type]) @ self.relation_attention[relation_name]  # 应用源类型和关系专属 Key 变换。
            projected_values = self.value_projections[source_type](features_by_type[source_type]) @ self.relation_message[relation_name]  # 应用源类型和关系专属 Value 变换。
            relation_scores = (order_queries[target_indices] * projected_keys[source_indices]).sum(dim=1) / math.sqrt(self.hidden_dim)  # 计算当前关系每条边的缩放点积分数。
            all_scores.append(relation_scores)  # 保存当前关系的逐边分数。
            all_values.append(projected_values[source_indices])  # 保存当前关系的逐边消息。
            all_targets.append(target_indices)  # 保存当前关系的目标订单编号。
            all_metadata.extend([(relation_name, int(source_index), int(target_index)) for source_index, target_index in zip(source_indices.tolist(), target_indices.tolist())])  # 保存可读边元数据。
        edge_scores = torch.cat(all_scores)  # 拼接两类关系的全部边分数。
        edge_values = torch.cat(all_values)  # 拼接两类关系的全部边消息。
        target_indices = torch.cat(all_targets)  # 拼接两类关系的全部目标编号。
        attention = torch.zeros_like(edge_scores)  # 创建逐边归一化注意力容器。
        for order_index in range(features_by_type["order"].shape[0]):  # 逐目标订单独立归一化异构入边。
            incoming_mask = target_indices == order_index  # 找出当前订单的用户边和设备边。
            attention[incoming_mask] = torch.softmax(edge_scores[incoming_mask], dim=0)  # 在当前订单全部异构入边上执行 softmax。
        messages = attention.unsqueeze(1) * edge_values  # 用关系注意力缩放每条异构消息。
        aggregated = torch.zeros(features_by_type["order"].shape[0], self.hidden_dim, device=edge_values.device)  # 创建每个订单的聚合表示。
        aggregated.index_add_(0, target_indices, messages)  # 按目标订单累加用户与设备消息。
        output = torch.relu(self.output_projection(aggregated) + self.residual_projection(features_by_type["order"]))  # 合并异构邻域与订单自身残差。
        return output, attention, edge_scores, target_indices, all_metadata  # 返回订单表示和逐边可解释中间量。
class HGTOrderClassifier(nn.Module):  # 定义异构图订单风险分类模型。
    def __init__(self):  # 创建手写 HGT 层和二分类头。
        super().__init__()  # 初始化父类以注册子模块。
        self.hgt = HGTOrderLayer()  # 创建关系感知的订单聚合层。
        self.classifier = nn.Linear(8, 2)  # 把订单图表示映射到正常与风险 logits。
    def forward(self, features_by_type, relations):  # 完成一次异构图前向传播。
        order_hidden, attention, scores, targets, metadata = self.hgt(features_by_type, relations)  # 聚合用户和设备关系消息。
        logits = self.classifier(order_hidden)  # 输出六个订单的二分类 logits。
        return logits, order_hidden, attention, scores, targets, metadata  # 返回预测和全部关键中间量。
hgt_model = HGTOrderClassifier()  # 实例化手写 HGT 订单分类器。
print(hgt_model)  # 展示节点类型与关系参数的真实结构。
print(f"可训练参数量={sum(parameter.numel() for parameter in hgt_model.parameters())}")  # 输出教学模型参数规模。

HGTOrderClassifier(
  (hgt): HGTOrderLayer(
    (query_projection): Linear(in_features=3, out_features=8, bias=False)
    (key_projections): ModuleDict(
      (user): Linear(in_features=3, out_features=8, bias=False)
      (device): Linear(in_features=3, out_features=8, bias=False)
    )
    (value_projections): ModuleDict(
      (user): Linear(in_features=3, out_features=8, bias=False)
      (device): Linear(in_features=3, out_features=8, bias=False)
    )
    (relation_attention): ParameterDict(
        (device__used__order): Parameter containing: [torch.FloatTensor of size 8x8]
        (user__places__order): Parameter containing: [torch.FloatTensor of size 8x8]
    )
    (relation_message): ParameterDict(
        (device__used__order): Parameter containing: [torch.FloatTensor of size 8x8]
        (user__places__order): Parameter containing: [torch.FloatTensor of size 8x8]
    )
    (output_projection): Linear(in_features=8, out_features=8, bias=True)
    (residual_projection): Linea

In [4]:
optimizer = torch.optim.Adam(hgt_model.parameters(), lr=0.03)  # 创建优化器更新类型、关系与分类参数。
loss_trace = []  # 保存异构图训练损失轨迹。
first_relation_gradient = 0.0  # 预留首轮设备关系矩阵梯度范数。
for epoch in range(301):  # 在六订单小图上执行三百零一次全图更新。
    optimizer.zero_grad()  # 清空上一轮累计梯度。
    logits, order_hidden, edge_attention, edge_scores, edge_targets, edge_metadata = hgt_model(node_features, edge_sets)  # 运行手写 HGT 前向传播。
    loss = F.cross_entropy(logits, order_labels)  # 计算订单风险分类交叉熵。
    loss.backward()  # 反向传播到节点类型与关系专属参数。
    if epoch == 0:  # 首轮记录真实关系矩阵梯度。
        first_relation_gradient = hgt_model.hgt.relation_attention["device__used__order"].grad.norm().item()  # 读取设备关系 attention 矩阵梯度范数。
    optimizer.step()  # 根据当前梯度更新 HGT 全部参数。
    loss_trace.append(loss.item())  # 保存当前轮训练损失。
    if epoch in [0, 50, 150, 300]:  # 选择关键轮次输出真实训练轨迹。
        current_accuracy = (logits.argmax(dim=1) == order_labels).float().mean().item()  # 计算当前订单分类准确率。
        print(f"epoch={epoch:03d} loss={loss.item():.4f} accuracy={current_accuracy:.1%}")  # 输出损失与准确率变化。
hgt_model.eval()  # 切换到评估模式生成稳定结果。
with torch.no_grad():  # 关闭评估阶段的梯度记录。
    final_logits, final_hidden, final_attention, final_scores, final_targets, final_metadata = hgt_model(node_features, edge_sets)  # 重新计算最终异构图结果。
hgt_probabilities = torch.softmax(final_logits, dim=1)[:, 1]  # 提取风险类别概率。
hgt_predictions = final_logits.argmax(dim=1)  # 选择概率最大的订单类别。
hgt_accuracy = (hgt_predictions == order_labels).float().mean().item()  # 计算 HGT 在同一六订单上的准确率。
print(f"首轮设备关系矩阵梯度范数={first_relation_gradient:.6f}")  # 输出非零梯度证明关系参数被真实训练。
print("逐订单异构入边注意力：")  # 标记即将展示的用户边与设备边权重。
for order_index, order_id in enumerate(order_ids):  # 逐订单展示两种关系的归一化权重。
    incoming_positions = torch.where(final_targets == order_index)[0]  # 找出当前订单的全部异构入边位置。
    readable_edges = [(final_metadata[position][0].split("__")[0], round(final_attention[position].item(), 4)) for position in incoming_positions.tolist()]  # 生成关系类型与注意力列表。
    print(f"{order_id}: {readable_edges}，权重和={final_attention[incoming_positions].sum().item():.4f}")  # 输出当前订单两类关系权重与总和。

epoch=000 loss=0.6992 accuracy=66.7%
epoch=050 loss=0.0000 accuracy=100.0%


epoch=150 loss=0.0000 accuracy=100.0%


epoch=300 loss=0.0000 accuracy=100.0%
首轮设备关系矩阵梯度范数=0.003994
逐订单异构入边注意力：
O001: [('user', 0.0009), ('device', 0.9991)]，权重和=1.0000
O002: [('user', 0.0218), ('device', 0.9782)]，权重和=1.0000
O003: [('user', 0.9977), ('device', 0.0023)]，权重和=1.0000
O004: [('user', 0.0147), ('device', 0.9853)]，权重和=1.0000
O005: [('user', 0.0033), ('device', 0.9967)]，权重和=1.0000
O006: [('user', 0.9943), ('device', 0.0057)]，权重和=1.0000


## 结果解读

O002、O004 的订单单点分数低，但用户拒付与设备共享风险高。逐订单表同时给出自身基线和 HGT 概率，避免只看一个总准确率。

In [5]:
print("订单   真实  基线  HGT风险概率  HGT预测  隐藏表示前3维")  # 打印逐订单异构分类结果表头。
for order_index, order_id in enumerate(order_ids):  # 遍历六个订单展示完整预测证据。
    hidden_preview = [round(value, 3) for value in final_hidden[order_index, :3].tolist()]  # 读取当前订单图表示前三维。
    print(f"{order_id}  {order_labels[order_index].item()}     {baseline_predictions[order_index].item()}     {hgt_probabilities[order_index]:.3f}       {hgt_predictions[order_index].item()}       {hidden_preview}")  # 输出当前订单的基线、HGT 与中间表示。
print(f"同数据准确率：订单单点基线={baseline_accuracy:.1%}，手写 HGT={hgt_accuracy:.1%}")  # 汇总两个方案的同口径指标。

订单   真实  基线  HGT风险概率  HGT预测  隐藏表示前3维
O001  1     1     1.000       1       [0.0, 0.0, 16.903]
O002  1     0     1.000       1       [0.0, 0.0, 16.62]
O003  0     0     0.000       0       [11.951, 0.0, 0.0]
O004  1     0     1.000       1       [0.0, 0.0, 15.617]
O005  1     1     1.000       1       [0.0, 0.0, 15.762]
O006  0     0     0.000       0       [11.34, 0.0, 0.0]
同数据准确率：订单单点基线=66.7%，手写 HGT=100.0%


## 失败案例：把全部图边放在一次全局 softmax

注意力应回答“对当前订单，用户和设备谁更重要”。如果在十二条边上做一次全局 softmax，O001 两条入边权重和远小于 1，而且会随其他订单数量变化。修复是按目标订单分组归一化。

In [6]:
wrong_global_attention = torch.softmax(final_scores, dim=0)  # 复现把全图十二条边一起归一化的错误实现。
example_order_index = 0  # 选择 O001 展示归一化口径错误。
example_positions = torch.where(final_targets == example_order_index)[0]  # 找出 O001 的用户边和设备边。
wrong_incoming_sum = wrong_global_attention[example_positions].sum().item()  # 计算错误全局 softmax 下 O001 入边权重和。
fixed_incoming_sum = final_attention[example_positions].sum().item()  # 计算正确按目标 softmax 下 O001 入边权重和。
print(f"错误全图 softmax：O001 入边权重和={wrong_incoming_sum:.4f}")  # 展示权重不再具有目标节点内概率含义。
print(f"修复按目标 softmax：O001 入边权重和={fixed_incoming_sum:.4f}")  # 展示用户和设备证据正确归一化为一。
print("修复结论：先拼接所有关系分数，再按 destination node 分组做 softmax。")  # 给出异构关系注意力的明确实现顺序。

错误全图 softmax：O001 入边权重和=0.2697
修复按目标 softmax：O001 入边权重和=1.0000
修复结论：先拼接所有关系分数，再按 destination node 分组做 softmax。


## 生产差距

线上 HGT 要处理更多节点/边类型、时间戳、邻居采样、关系 schema 版本和新类型冷启动。全图训练需替换为分层采样或图分区，还要监控类型不平衡、节点度数漂移和调查标签延迟。注意力权重描述模型聚合，不等于关系因果性。

## 最小回归测试

In [7]:
assert len(order_ids) >= 5  # 保证案例包含足够多的真实可读订单节点。
assert loss_trace[-1] < loss_trace[0]  # 保证真实反向传播使 HGT 损失下降。
assert first_relation_gradient > 0.0  # 保证关系专属 attention 矩阵获得非零梯度。
assert hgt_accuracy > baseline_accuracy  # 保证异构邻域在同数据上修正规则错误。
assert hgt_accuracy == 1.0  # 保证教学模型拟合六订单受控小图。
assert torch.allclose(torch.stack([final_attention[final_targets == index].sum() for index in range(len(order_ids))]), torch.ones(len(order_ids)), atol=1e-5)  # 保证每个订单的异构入边权重和为一。
assert wrong_incoming_sum < 0.5 and abs(fixed_incoming_sum - 1.0) < 1e-5  # 保证全局 softmax 失败与按目标修复可复现。
print("回归测试通过：关系参数、异构注意力和目标分组归一化均符合预期。")  # 输出集中断言的最终验收结论。

回归测试通过：关系参数、异构注意力和目标分组归一化均符合预期。
